# Thai Air Intelligence — Production Dual-Model PM2.5 v5.6.2

This Colab notebook trains and atomically promotes the two deployed model families:

- **Numeric PM2.5:** 20 province-local residual LightGBM regressors for D+1 through D+7.
- **Air-quality class:** one pooled Random Forest classifier for validated D+1 probabilities.
- **Fallback:** the recent seven-day observed mean when a serving artifact is unavailable.

The residual regressors learn a correction to Persistence. Their validation-selected correction weights are conservatively shrunk and frozen before the untouched Test window. Promotion requires at least 4.5% D+1 MAE Skill globally and in every province, plus the reviewed pooled classification gates. Native and portable tree predictions must match before registration.

Training state is checkpointed after the chronological split and after each model task. If Colab restarts after a long training cell, rerun Configuration through Pipeline Imports once, then rerun the interrupted cell; it restores the latest compatible checkpoint instead of rebuilding completed models.

Promotion is all-or-nothing: 20 regression rows and 20 classification rows activate in one database transaction. The notebook then generates a seven-day forecast and verifies 140 rows, including 20 D+1 rows using the new active Random Forest artifact.

Required Colab Secrets: `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY`, `ML_SECRET`, and either `ML_FORECAST_URL` or `WEBSITE_URL`.


In [ ]:
# 1. Reviewed Production configuration — v5.6.2
REGISTER = False
ACTIVATE = False
RUN_FORECAST = False
PRODUCTION_APPROVAL = "APPROVED_RESIDUAL_LIGHTGBM_V5_6_2"
APPROVED_CODE_SHA = "4d271c4114780b755813a2937a7c9db72b5185e5"

PROVINCE = "all"
MINIMUM_ROWS = 180
CV_SPLITS = 3
ARTIFACT_DIRECTORY = "training/artifacts"
ALLOWED_SOURCES = {"open-meteo"}
FORECAST_HORIZON_DAYS = 7
PRODUCTION_REQUIRED_PROVINCES = 20
REGRESSION_MINIMUM_SKILL = 0.045
CHECKPOINT_FILE = "/content/pm25_v5_6_2_checkpoint.joblib"

if not (REGISTER and ACTIVATE and RUN_FORECAST):
    raise ValueError("v5.6.2 Production requires REGISTER=True, ACTIVATE=True, RUN_FORECAST=True")
if PRODUCTION_APPROVAL != "APPROVED_RESIDUAL_LIGHTGBM_V5_6_2":
    raise ValueError("Production approval token is missing")
if len(APPROVED_CODE_SHA) != 40:
    raise ValueError("APPROVED_CODE_SHA must be a reviewed 40-character commit SHA")
if PROVINCE != "all" or PRODUCTION_REQUIRED_PROVINCES != 20:
    raise ValueError("Production activation requires the complete 20-province pool")
if MINIMUM_ROWS < 180:
    raise ValueError("Do not lower MINIMUM_ROWS below the reviewed 180-day gate")
print({
    "notebook_version": "5.6.2",
    "mode": "production_register_activate_forecast",
    "approved_code_sha": APPROVED_CODE_SHA,
    "regression_minimum_skill": REGRESSION_MINIMUM_SKILL,
})


In [ ]:
# 2. Fetch the reviewed code and install only the missing training packages
# Move to /content first so rerunning this cell never deletes Colab's current working directory.
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_DIRECTORY = Path("/content/THAI-AIR-INTELLIGENCE-LITE")
os.chdir("/content")
shutil.rmtree(REPOSITORY_DIRECTORY, ignore_errors=True)

subprocess.run(
    ["git", "clone", "--filter=blob:none", "https://github.com/kzabCde/THAI-AIR-INTELLIGENCE-LITE.git", str(REPOSITORY_DIRECTORY)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPOSITORY_DIRECTORY), "checkout", "--detach", APPROVED_CODE_SHA],
    check=True,
)
checked_out_sha = subprocess.check_output(
    ["git", "-C", str(REPOSITORY_DIRECTORY), "rev-parse", "HEAD"],
    text=True,
).strip()
if checked_out_sha != APPROVED_CODE_SHA:
    raise RuntimeError(f"Expected {APPROVED_CODE_SHA}, checked out {checked_out_sha}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade-strategy",
        "only-if-needed",
        "supabase==2.31.0",
        "lightgbm==4.6.0",
        "joblib==1.4.2",
    ],
    check=True,
)

os.chdir(REPOSITORY_DIRECTORY)
repository_path = str(REPOSITORY_DIRECTORY)
sys.path = [path for path in sys.path if path != repository_path]
sys.path.insert(0, repository_path)
for module_name in list(sys.modules):
    if module_name in {"training", "api"} or module_name.startswith(("training.", "api.")):
        del sys.modules[module_name]
importlib.invalidate_caches()
print({"repository": repository_path, "approved_code_sha": checked_out_sha})

In [ ]:
# 3. Scientific-stack and native-model preflight
import platform
import lightgbm
import numpy as np
import pandas as pd
import sklearn
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestClassifier

probe_X = np.asarray([[0.0], [1.0], [2.0], [3.0]], dtype=float)
LGBMRegressor(n_estimators=2, verbose=-1).fit(probe_X, probe_X[:, 0]).predict(probe_X[:1])
RandomForestClassifier(n_estimators=2, random_state=42).fit(probe_X, [1, 1, 2, 2]).predict_proba(probe_X[:1])
print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "lightgbm": lightgbm.__version__,
})
print("Environment ABI check passed")

In [ ]:
# 4. Load server-side Supabase and forecast secrets without printing their values
import os
from google.colab import userdata

def _required_secret(*names):
    for name in names:
        try:
            value = (userdata.get(name) or "").strip()
        except Exception:
            value = ""
        if value:
            return value
    raise ValueError(f"Missing Colab Secret; add one of: {', '.join(names)}")

os.environ["SUPABASE_URL"] = _required_secret("SUPABASE_URL")
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = _required_secret("SUPABASE_SERVICE_ROLE_KEY")
ML_SECRET = _required_secret("ML_SECRET")
ML_FORECAST_URL = _required_secret("ML_FORECAST_URL", "WEBSITE_URL").rstrip("/")

if not os.environ["SUPABASE_URL"].startswith("https://"):
    raise ValueError("SUPABASE_URL must start with https://")
if os.environ["SUPABASE_SERVICE_ROLE_KEY"].lower().startswith(("sb_publishable_", "sb_anon_")):
    raise ValueError("Use a server-side service-role/secret key, not a public key")
if not ML_FORECAST_URL.startswith("https://"):
    raise ValueError("ML_FORECAST_URL or WEBSITE_URL must start with https://")
print("Supabase and forecast endpoint secrets loaded")


In [ ]:
# 5. Import the exact residual-regression and pooled-classification pipeline
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

from training.dual_model_config import (
    FALLBACK_MODEL_NAME,
    FALLBACK_STRATEGY,
    FALLBACK_WINDOW_DAYS,
    POOLED_FEATURE_COLUMNS,
    POOLED_FEATURE_PROVENANCE,
    POOLED_FEATURE_VERSION,
    POOLED_PROVINCE_IDS,
    PipelineConfig,
)
from training.pm25_classes import CLASS_IDS, THRESHOLD_VERSION
from training.train_dual_models import (
    OBSERVED_VIEW,
    _json_safe,
    fetch_observed_rows,
    filter_training_rows,
)
from training.train_pooled_models import (
    CLASSIFICATION_MODEL_NAME,
    DIRECT_HORIZONS,
    REGRESSION_MODEL_NAME,
    PooledResult,
    build_pooled_examples,
    build_registry_rows,
    fetch_province_metadata,
    get_client,
    pooled_chronological_split,
    save_artifacts,
    train_classification,
    train_regression,
    upload_and_register,
)

config = PipelineConfig(
    minimum_rows=MINIMUM_ROWS,
    cv_splits=CV_SPLITS,
    artifact_directory=Path(ARTIFACT_DIRECTORY),
)
config.validate()
selected_provinces = tuple(POOLED_PROVINCE_IDS) if PROVINCE == "all" else (PROVINCE,)
if any(province_id not in POOLED_PROVINCE_IDS for province_id in selected_provinces):
    raise ValueError(f"Unknown province selection: {selected_provinces}")
print({
    "regression": REGRESSION_MODEL_NAME,
    "classification": CLASSIFICATION_MODEL_NAME,
    "fallback": {
        "model": FALLBACK_MODEL_NAME,
        "strategy": FALLBACK_STRATEGY,
        "window_days": FALLBACK_WINDOW_DAYS,
    },
    "features": len(POOLED_FEATURE_COLUMNS),
    "feature_version": POOLED_FEATURE_VERSION,
    "provinces": len(selected_provinces),
    "direct_horizons": list(DIRECT_HORIZONS),
    "register": REGISTER,
    "activate": ACTIVATE,
})

import joblib

CHECKPOINT_PATH = Path(CHECKPOINT_FILE)
CHECKPOINT_SCHEMA = "pm25-residual-v5.6.2"

def _save_checkpoint(stage, **state):
    payload = {
        "schema": CHECKPOINT_SCHEMA,
        "approved_code_sha": APPROVED_CODE_SHA,
        "stage": stage,
        **state,
    }
    joblib.dump(payload, CHECKPOINT_PATH)
    print({"checkpoint": str(CHECKPOINT_PATH), "stage": stage})

def _restore_checkpoint(required):
    if not CHECKPOINT_PATH.exists():
        raise RuntimeError(
            "No compatible checkpoint exists. Run Configuration through the chronological split first."
        )
    payload = joblib.load(CHECKPOINT_PATH)
    if payload.get("schema") != CHECKPOINT_SCHEMA:
        raise RuntimeError("Checkpoint schema does not match v5.6.2")
    if payload.get("approved_code_sha") != APPROVED_CODE_SHA:
        raise RuntimeError("Checkpoint belongs to a different reviewed code commit")
    missing = [name for name in required if name not in payload]
    if missing:
        raise RuntimeError(f"Checkpoint stage {payload.get('stage')} is missing: {missing}")
    globals().update({name: payload[name] for name in required})
    print({"restored_checkpoint": str(CHECKPOINT_PATH), "stage": payload.get("stage")})


In [ ]:
# 6. Fetch trusted database rows and extend them in memory with multi-season Open-Meteo history
import time
from itertools import islice

import requests


AIR_ARCHIVE_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
WEATHER_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
ARCHIVE_WEATHER_VARIABLES = (
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "surface_pressure",
    "precipitation",
)


def _batches(values, size):
    iterator = iter(values)
    while True:
        batch = tuple(islice(iterator, size))
        if not batch:
            return
        yield batch


def _date_chunks(start, end, days):
    current = pd.Timestamp(start).normalize()
    end = pd.Timestamp(end).normalize()
    while current <= end:
        chunk_end = min(end, current + pd.Timedelta(days=days - 1))
        yield current, chunk_end
        current = chunk_end + pd.Timedelta(days=1)


def _request_json(url, params, attempts=5):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=ARCHIVE_REQUEST_TIMEOUT_SECONDS,
                headers={"User-Agent": "thai-air-intelligence-shadow-lab-v5.2"},
            )
            response.raise_for_status()
            payload = response.json()
            if isinstance(payload, dict) and payload.get("error"):
                raise RuntimeError(str(payload.get("reason") or payload))
            return payload
        except Exception as exc:
            last_error = exc
            if attempt == attempts:
                break
            time.sleep(min(20.0, 2.0 ** (attempt - 1)))
    raise RuntimeError(f"Open-Meteo archive request failed: {last_error}")


def _locations(payload, expected):
    locations = payload if isinstance(payload, list) else [payload]
    if len(locations) != expected:
        raise RuntimeError(f"Expected {expected} Open-Meteo locations, received {len(locations)}")
    return locations


def _daily_air(location, province_id):
    hourly = location.get("hourly") or {}
    frame = pd.DataFrame({
        "date": pd.to_datetime(hourly.get("time") or [], errors="raise").normalize(),
        "pm25": pd.to_numeric(hourly.get("pm2_5") or [], errors="coerce"),
    })
    frame["province_id"] = province_id
    return frame.groupby(["province_id", "date"], as_index=False).agg(
        pm25_mean=("pm25", "mean"),
        trusted_hours=("pm25", "count"),
    )


def _daily_weather(location, province_id):
    hourly = location.get("hourly") or {}
    frame = pd.DataFrame({"date": pd.to_datetime(hourly.get("time") or [], errors="raise").normalize()})
    for column in ARCHIVE_WEATHER_VARIABLES:
        frame[column] = pd.to_numeric(hourly.get(column) or [], errors="coerce")
    frame["province_id"] = province_id
    return frame.groupby(["province_id", "date"], as_index=False).agg(
        temp_mean=("temperature_2m", "mean"),
        humidity_mean=("relative_humidity_2m", "mean"),
        wind_speed_mean=("wind_speed_10m", "mean"),
        pressure_mean=("surface_pressure", "mean"),
        precip_total=("precipitation", "sum"),
    )


def _fetch_archive_daily(province_metadata, start, end):
    rows = []
    metadata_rows = province_metadata.sort_values("province_id").to_dict("records")
    total_chunks = sum(1 for _ in _date_chunks(start, end, ARCHIVE_CHUNK_DAYS))
    total_batches = math.ceil(len(metadata_rows) / ARCHIVE_PROVINCE_BATCH_SIZE)
    done = 0
    for chunk_start, chunk_end in _date_chunks(start, end, ARCHIVE_CHUNK_DAYS):
        for province_batch in _batches(metadata_rows, ARCHIVE_PROVINCE_BATCH_SIZE):
            coordinate_params = {
                "latitude": ",".join(str(row["lat"]) for row in province_batch),
                "longitude": ",".join(str(row["lon"]) for row in province_batch),
                "timezone": "Asia/Bangkok",
                "start_date": chunk_start.date().isoformat(),
                "end_date": chunk_end.date().isoformat(),
            }
            air_payload = _request_json(AIR_ARCHIVE_URL, {
                **coordinate_params,
                "hourly": "pm2_5",
                "domains": "cams_global",
            })
            weather_payload = _request_json(WEATHER_ARCHIVE_URL, {
                **coordinate_params,
                "hourly": ",".join(ARCHIVE_WEATHER_VARIABLES),
            })
            air_locations = _locations(air_payload, len(province_batch))
            weather_locations = _locations(weather_payload, len(province_batch))
            for index, province in enumerate(province_batch):
                air = _daily_air(air_locations[index], province["province_id"])
                weather = _daily_weather(weather_locations[index], province["province_id"])
                daily = air.merge(weather, on=["province_id", "date"], how="inner")
                rows.append(daily)
            done += 1
            print(
                f"[Archive] {done}/{total_chunks * total_batches} "
                f"{chunk_start.date()}..{chunk_end.date()}",
                flush=True,
            )
    if not rows:
        return pd.DataFrame()
    archive = pd.concat(rows, ignore_index=True)
    archive = archive[archive["trusted_hours"] >= 18].copy()
    archive["trusted_sources"] = [["open-meteo"] for _ in range(len(archive))]
    archive["data_origin"] = "open-meteo-cams-and-weather-archive"
    return archive


def _exact_lag(frame, days):
    result = np.full(len(frame), np.nan, dtype=float)
    for _, positions in frame.groupby("province_id", sort=False).groups.items():
        positions = list(positions)
        province = frame.loc[positions]
        lookup = province.set_index("date")["pm25_mean"]
        result[positions] = (
            province["date"] - pd.Timedelta(days=days)
        ).map(lookup).to_numpy(dtype=float)
    return result


def _rebuild_leakage_safe_daily_features(frame, province_metadata):
    rebuilt = frame.sort_values(["province_id", "date"]).drop_duplicates(
        ["province_id", "date"], keep="last"
    ).reset_index(drop=True)
    for days in range(1, 8):
        rebuilt[f"_pm25_lag_{days}d"] = _exact_lag(rebuilt, days)
    rebuilt["pm25_lag_1d"] = rebuilt["_pm25_lag_1d"]
    rebuilt["pm25_lag_3d"] = rebuilt["_pm25_lag_3d"]
    rebuilt["pm25_lag_6d"] = rebuilt["_pm25_lag_6d"]
    rebuilt["pm25_lag_7d"] = rebuilt["_pm25_lag_7d"]
    rebuilt["pm25_roll3"] = rebuilt[[
        "pm25_mean", "_pm25_lag_1d", "_pm25_lag_2d"
    ]].mean(axis=1, skipna=False)
    rebuilt["pm25_roll7"] = rebuilt[[
        "pm25_mean", *[f"_pm25_lag_{days}d" for days in range(1, 7)]
    ]].mean(axis=1, skipna=False)

    coordinates = province_metadata.set_index("province_id")[["lat", "lon"]]
    neighbor_ids = {}
    for province_id in coordinates.index:
        delta = coordinates - coordinates.loc[province_id]
        distance = np.square(delta["lat"]) + np.square(delta["lon"])
        neighbor_ids[province_id] = tuple(distance.drop(province_id).nsmallest(3).index)
    pivot = rebuilt.pivot(index="date", columns="province_id", values="pm25_mean")
    regional_mean = pivot.mean(axis=1)
    neighbor_lookup = {}
    for province_id, neighbors in neighbor_ids.items():
        neighbor_lookup[province_id] = pivot.reindex(columns=list(neighbors)).mean(axis=1)
    rebuilt["regional_pm25_avg"] = rebuilt["date"].map(regional_mean)
    rebuilt["neighbor_pm25_avg"] = [
        neighbor_lookup[province_id].get(date, np.nan)
        for province_id, date in zip(rebuilt["province_id"], rebuilt["date"], strict=True)
    ]
    rebuilt["month"] = rebuilt["date"].dt.month.astype(int)
    rebuilt["day_of_week"] = rebuilt["date"].dt.dayofweek.astype(int)
    rebuilt["is_burning_season"] = rebuilt["month"].isin((1, 2, 3, 4)).astype(float)
    rebuilt["is_dry_season"] = rebuilt["month"].isin((11, 12, 1, 2, 3, 4)).astype(float)
    day_of_year = rebuilt["date"].dt.dayofyear.to_numpy(dtype=float)
    rebuilt["day_of_year_sin"] = np.sin(2.0 * np.pi * day_of_year / 365.25)
    rebuilt["day_of_year_cos"] = np.cos(2.0 * np.pi * day_of_year / 365.25)
    return rebuilt.drop(columns=[f"_pm25_lag_{days}d" for days in range(1, 8)])


sb = get_client()
metadata_all = fetch_province_metadata(sb, context_provinces)
raw_database = fetch_observed_rows(sb, context_provinces)
database_observed = filter_training_rows(raw_database, ALLOWED_SOURCES).copy()
database_observed["data_origin"] = "supabase-training-daily-summary-v2"

archive_daily = pd.DataFrame()
database_first_date = pd.Timestamp(database_observed["date"].min()).normalize()
archive_start = pd.Timestamp(ARCHIVE_START_DATE)
archive_end = database_first_date - pd.Timedelta(days=1)
if USE_IN_MEMORY_ARCHIVE and archive_start <= archive_end:
    archive_daily = _fetch_archive_daily(metadata_all, archive_start, archive_end)

combined = pd.concat([archive_daily, database_observed], ignore_index=True, sort=False)
combined = _rebuild_leakage_safe_daily_features(combined, metadata_all)
observed = combined[combined["province_id"].isin(selected_provinces)].copy()
metadata = metadata_all[metadata_all["province_id"].isin(selected_provinces)].copy()

origin_counts = (
    observed.groupby(["province_id", "data_origin"]).size().unstack(fill_value=0)
)
quality = (
    observed.groupby("province_id", as_index=False)
    .agg(
        usable_days=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        minimum_trusted_hours=("trusted_hours", "min"),
        mean_pm25=("pm25_mean", "mean"),
    )
    .sort_values("province_id")
)
quality["annual_cycles"] = quality["usable_days"] / 365.25
quality["has_multi_season_history"] = quality["annual_cycles"].ge(MINIMUM_ANNUAL_CYCLES)
quality = quality.merge(origin_counts.reset_index(), on="province_id", how="left")
display(quality)

archive_audit = {
    "enabled": USE_IN_MEMORY_ARCHIVE,
    "start_date": archive_start.date().isoformat(),
    "end_date": archive_end.date().isoformat(),
    "air_source": "Open-Meteo CAMS global model-derived PM2.5",
    "weather_source": "Open-Meteo Historical Weather API",
    "timezone": "Asia/Bangkok",
    "database_writes": 0,
    "database_overlap_policy": "database row wins; archive stops before first database date",
}
assert set(quality["province_id"]) == set(selected_provinces)
assert quality["minimum_trusted_hours"].ge(18).all()
if USE_IN_MEMORY_ARCHIVE and not quality["has_multi_season_history"].all():
    failed = quality.loc[~quality["has_multi_season_history"], "province_id"].tolist()
    raise RuntimeError(f"Multi-season history gate failed: {failed}")


In [ ]:
# 7. Build pooled D+1..D+7 examples from actual future PM2.5
examples = build_pooled_examples(observed, metadata)
example_counts = (
    examples.groupby(["province_id", "forecast_horizon_days"])
    .size()
    .unstack(fill_value=0)
)
class_distribution = (
    examples.groupby(["province_id", "target_air_quality_class"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CLASS_IDS, fill_value=0)
)
print("Pooled examples:", len(examples))
display(example_counts)
display(class_distribution)
assert examples.loc[:, POOLED_FEATURE_COLUMNS].notna().all(axis=None)
assert examples["target_pm25"].notna().all()

In [ ]:
# 8. Purged chronological split — every province on one date stays together
split = pooled_chronological_split(examples, config)
split_frames = {"train": split.train, "validation": split.validation, "test": split.test}
split_rows = []
date_sets = {}
for split_name, frame in split_frames.items():
    date_sets[split_name] = set(frame["date"].unique())
    split_rows.append({
        "split": split_name,
        "rows": len(frame),
        "origin_dates": frame["date"].nunique(),
        "first_origin": frame["date"].min(),
        "last_origin": frame["date"].max(),
        "last_target": frame["target_date"].max(),
        "provinces": frame["province_id"].nunique(),
    })
display(pd.DataFrame(split_rows))
print("Embargo dates removed:", len(split.dropped_embargo_dates), split.dropped_embargo_dates)
assert date_sets["train"].isdisjoint(date_sets["validation"])
assert date_sets["train"].isdisjoint(date_sets["test"])
assert date_sets["validation"].isdisjoint(date_sets["test"])
assert split.train["target_date"].max() < split.validation["date"].min()
assert split.validation["target_date"].max() < split.test["date"].min()

_save_checkpoint(
    "split",
    split=split,
    config=config,
    selected_provinces=selected_provinces,
)


In [ ]:
if "_restore_checkpoint" not in globals():
    raise RuntimeError(
        "Session restarted. Rerun Configuration through Pipeline Imports, then rerun this cell."
    )
if "split" not in globals():
    _restore_checkpoint(("split", "config", "selected_provinces"))

# 9. Train province-local residual LightGBM models and evaluate untouched D+1 evidence
regression = train_regression(split, config)
print(json.dumps(_json_safe({
    "model": REGRESSION_MODEL_NAME,
    "global_eligible": regression.global_eligible,
    "global_reasons": regression.global_reasons,
    "validation_metrics": regression.validation_metrics,
    "test_metrics": regression.test_metrics,
}), ensure_ascii=False, indent=2))
regression_by_province = pd.DataFrame([
    {"province_id": province_id, **metrics}
    for province_id, metrics in regression.province_metrics.items()
]).sort_values("province_id")
display(regression_by_province[[
    "province_id", "mae", "rmse", "r2",
    "skill_vs_persistence", "eligible"
]])

_save_checkpoint(
    "regression",
    split=split,
    config=config,
    selected_provinces=selected_provinces,
    regression=regression,
)


In [ ]:
if "_restore_checkpoint" not in globals():
    raise RuntimeError(
        "Session restarted. Rerun Configuration through Pipeline Imports, then rerun this cell."
    )
if any(name not in globals() for name in ("split", "config", "selected_provinces", "regression")):
    _restore_checkpoint(("split", "config", "selected_provinces", "regression"))

# 10. Train pooled Random Forest classification and evaluate class imbalance
classification = train_classification(split, config)
metrics = classification.test_metrics
print(json.dumps(_json_safe({
    "model": CLASSIFICATION_MODEL_NAME,
    "global_eligible": classification.global_eligible,
    "global_reasons": classification.global_reasons,
    "accuracy": metrics.get("accuracy"),
    "balanced_accuracy": metrics.get("balanced_accuracy"),
    "macro_f1": metrics.get("macro_f1"),
    "brier_score": metrics.get("brier_score"),
    "expected_calibration_error": metrics.get("expected_calibration_error"),
}), ensure_ascii=False, indent=2))
per_class = pd.DataFrame([
    {"class_id": class_id, **metrics.get("per_class", {}).get(str(class_id), {})}
    for class_id in CLASS_IDS
])
display(per_class)
display(pd.DataFrame(
    metrics.get("confusion_matrix", []),
    index=[f"actual_{class_id}" for class_id in CLASS_IDS],
    columns=[f"predicted_{class_id}" for class_id in CLASS_IDS],
))

_save_checkpoint(
    "classification",
    split=split,
    config=config,
    selected_provinces=selected_provinces,
    regression=regression,
    classification=classification,
)


In [ ]:
if "_restore_checkpoint" not in globals():
    raise RuntimeError(
        "Session restarted. Rerun Configuration through Pipeline Imports, then rerun this cell."
    )
if any(name not in globals() for name in (
    "split", "config", "selected_provinces", "regression", "classification"
)):
    _restore_checkpoint((
        "split", "config", "selected_provinces", "regression", "classification"
    ))

# 11. Independent province/task eligibility and fallback plan
eligibility = pd.DataFrame([
    {
        "province_id": province_id,
        "regression_eligible": bool(regression.province_metrics[province_id]["eligible"]),
        "regression_skill": regression.province_metrics[province_id].get("skill_vs_persistence"),
        "classification_eligible": bool(classification.province_metrics[province_id]["eligible"]),
        "classification_macro_f1": classification.province_metrics[province_id].get("macro_f1"),
        "forecast_class_source": (
            "random_forest_classifier"
            if classification.province_metrics[province_id]["eligible"]
            else "lightgbm_regression_threshold"
            if regression.province_metrics[province_id]["eligible"]
            else "recent_mean_7d_fallback"
        ),
    }
    for province_id in selected_provinces
]).sort_values("province_id")
display(eligibility)

if len(eligibility) != PRODUCTION_REQUIRED_PROVINCES:
    raise RuntimeError(f"Expected 20 provinces, found {len(eligibility)}")
if not regression.global_eligible or not eligibility["regression_eligible"].all():
    failed = eligibility.loc[~eligibility["regression_eligible"], "province_id"].tolist()
    raise RuntimeError(f"Regression is not deployable for every province: {failed}")
if eligibility["regression_skill"].min() < REGRESSION_MINIMUM_SKILL:
    raise RuntimeError("Regression minimum province Skill is below 4.5%")
if not classification.global_eligible or not eligibility["classification_eligible"].all():
    failed = eligibility.loc[~eligibility["classification_eligible"], "province_id"].tolist()
    raise RuntimeError(f"Classification is not deployable for every province: {failed}")

_save_checkpoint(
    "eligibility",
    split=split,
    config=config,
    selected_provinces=selected_provinces,
    regression=regression,
    classification=classification,
    eligibility=eligibility,
)


In [ ]:
if "_restore_checkpoint" not in globals():
    raise RuntimeError(
        "Session restarted. Rerun Configuration through Pipeline Imports, then rerun this cell."
    )
if any(name not in globals() for name in (
    "split", "config", "selected_provinces", "regression", "classification", "eligibility"
)):
    _restore_checkpoint((
        "split", "config", "selected_provinces", "regression", "classification", "eligibility"
    ))

# 12. Save exact native + portable artifacts and a reproducible run summary
run_id = str(uuid.uuid4())
audit = {
    "strategy": "pooled_split_local_residual_regression_and_pooled_classification",
    "pool_provinces": list(selected_provinces),
    "feature_version": POOLED_FEATURE_VERSION,
    "feature_provenance": POOLED_FEATURE_PROVENANCE,
    "target_source": "observed future PM2.5",
    "target_horizons": list(DIRECT_HORIZONS),
    "final_test_untouched_during_tuning": True,
    "same_date_same_partition": True,
    "embargo_days": max(DIRECT_HORIZONS),
    "dropped_embargo_dates": split.dropped_embargo_dates,
    "rows": {
        "train": len(split.train),
        "validation": len(split.validation),
        "test": len(split.test),
    },
}
registry_rows = build_registry_rows(
    run_id, selected_provinces, split, regression, classification, audit
)
result = PooledResult(
    run_id, selected_provinces, split, regression, classification, registry_rows, audit
)
artifacts = save_artifacts(result, config.artifact_directory, config)
run_summary = {
    "run_id": run_id,
    "mode": "register" if REGISTER else "shadow",
    "activate": ACTIVATE,
    "fallback": {
        "model": FALLBACK_MODEL_NAME,
        "strategy": FALLBACK_STRATEGY,
        "window_days": FALLBACK_WINDOW_DAYS,
        "used_for_provinces": eligibility.loc[
            ~eligibility["regression_eligible"], "province_id"
        ].tolist(),
    },
    "regression": {
        "model": REGRESSION_MODEL_NAME,
        "global_eligible": regression.global_eligible,
        "eligible_provinces": eligibility.loc[eligibility["regression_eligible"], "province_id"].tolist(),
        "metrics": regression.test_metrics,
    },
    "classification": {
        "model": CLASSIFICATION_MODEL_NAME,
        "global_eligible": classification.global_eligible,
        "eligible_provinces": eligibility.loc[eligibility["classification_eligible"], "province_id"].tolist(),
        "metrics": classification.test_metrics,
    },
    "audit": audit,
    "created_at": datetime.now(timezone.utc).isoformat(),
}
summary_path = config.artifact_directory / run_id / "run_summary.json"
summary_path.write_text(
    json.dumps(_json_safe(run_summary), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("RUN_SUMMARY:", summary_path)
print(json.dumps(_json_safe(run_summary), ensure_ascii=False, indent=2))

if len(registry_rows) != PRODUCTION_REQUIRED_PROVINCES * 2:
    raise RuntimeError(f"Expected 40 registry rows, found {len(registry_rows)}")
if set(artifacts["regression"]) != set(selected_provinces):
    raise RuntimeError("Expected one residual LightGBM artifact for every province")
if set(artifacts["classification"]) != {"pooled"}:
    raise RuntimeError("Expected exactly one pooled Random Forest artifact")

_save_checkpoint(
    "artifacts",
    split=split,
    config=config,
    selected_provinces=selected_provinces,
    regression=regression,
    classification=classification,
    eligibility=eligibility,
    run_id=run_id,
    audit=audit,
    registry_rows=registry_rows,
    result=result,
    artifacts=artifacts,
    run_summary=run_summary,
    summary_path=summary_path,
)


In [ ]:
if "_restore_checkpoint" not in globals():
    raise RuntimeError(
        "Session restarted. Rerun Configuration through Pipeline Imports, then rerun this cell."
    )
if any(name not in globals() for name in ("result", "artifacts", "run_id")):
    _restore_checkpoint((
        "config", "selected_provinces", "regression", "classification", "eligibility",
        "run_id", "result", "artifacts", "run_summary", "summary_path"
    ))
if "sb" not in globals():
    sb = get_client()

# 13. Optional promotion — uploads/registers candidates; activation is atomic across both tasks
if REGISTER:
    upload_and_register(sb, result, artifacts, activate=ACTIVATE)
    print({"registered": True, "activate_requested": ACTIVATE, "run_id": run_id})
else:
    print("REGISTER is False; no Storage, model_registry, activation, or forecast writes occurred.")

_save_checkpoint(
    "activated",
    config=config,
    selected_provinces=selected_provinces,
    regression=regression,
    classification=classification,
    eligibility=eligibility,
    run_id=run_id,
    result=result,
    artifacts=artifacts,
    run_summary=run_summary,
    summary_path=summary_path,
)


In [ ]:
if "_restore_checkpoint" not in globals():
    raise RuntimeError(
        "Session restarted. Rerun Configuration through Pipeline Imports, then rerun this cell."
    )
if "run_id" not in globals():
    _restore_checkpoint(("run_id", "selected_provinces", "run_summary", "summary_path"))
if "sb" not in globals():
    sb = get_client()

# 14. Read-only verification: at most one active model per province and task
active_rows = (
    sb.table("model_registry")
    .select("province_id,task_type,model_name,model_family,run_id,eligibility_status,is_active")
    .eq("is_active", True)
    .order("province_id")
    .order("task_type")
    .execute()
    .data
    or []
)
active_models = pd.DataFrame(active_rows)
display(active_models)
if not active_models.empty:
    active_counts = active_models.groupby(["province_id", "task_type"]).size()
    assert int(active_counts.max()) <= 1

new_active_rows = [
    row for row in active_rows
    if row["run_id"] == run_id
    and row["province_id"] in selected_provinces
    and row["task_type"] in {"regression", "classification"}
]
if len(new_active_rows) != PRODUCTION_REQUIRED_PROVINCES * 2:
    raise RuntimeError(f"Expected 40 active rows for run {run_id}, found {len(new_active_rows)}")
new_active_frame = pd.DataFrame(new_active_rows)
task_counts = new_active_frame.groupby("task_type")["province_id"].nunique().to_dict()
if task_counts != {"classification": 20, "regression": 20}:
    raise RuntimeError(f"Atomic activation readback is incomplete: {task_counts}")


In [ ]:
# 15. Generate a fresh seven-day forecast and verify both active model tasks
import requests

if not RUN_FORECAST:
    raise RuntimeError("RUN_FORECAST must remain True for the approved Production run")
if "_restore_checkpoint" not in globals():
    raise RuntimeError(
        "Session restarted. Rerun Configuration through Pipeline Imports, then rerun this cell."
    )
if any(name not in globals() for name in (
    "run_id", "selected_provinces", "run_summary", "summary_path", "config"
)):
    _restore_checkpoint((
        "run_id", "selected_provinces", "run_summary", "summary_path", "config"
    ))
if "sb" not in globals():
    sb = get_client()

active_for_run = (
    sb.table("model_registry")
    .select("province_id,task_type,model_name,run_id,is_active")
    .eq("run_id", run_id)
    .eq("is_active", True)
    .execute()
    .data
    or []
)
if len(active_for_run) != PRODUCTION_REQUIRED_PROVINCES * 2:
    raise RuntimeError(f"Expected 40 active rows for run {run_id}, found {len(active_for_run)}")

forecast_started_at = datetime.now(timezone.utc).isoformat()
forecast_response = requests.post(
    f"{ML_FORECAST_URL}/api/ml/forecast",
    headers={
        "Authorization": f"Bearer {ML_SECRET}",
        "Content-Type": "application/json",
    },
    json={"horizon": FORECAST_HORIZON_DAYS},
    timeout=180,
)
if forecast_response.status_code != 200:
    raise RuntimeError(
        f"Forecast endpoint returned HTTP {forecast_response.status_code}: "
        f"{forecast_response.text[:500]}"
    )
forecast_response_json = forecast_response.json()
if not forecast_response_json.get("ok"):
    raise RuntimeError(f"Forecast endpoint did not confirm success: {forecast_response_json}")

forecast_runs = (
    sb.table("forecast_runs")
    .select("run_id,status,forecast_at,completed_at,configuration,error_message")
    .gte("forecast_at", forecast_started_at)
    .order("forecast_at", desc=True)
    .limit(1)
    .execute()
    .data
    or []
)
if len(forecast_runs) != 1 or forecast_runs[0]["status"] != "success":
    raise RuntimeError(f"Latest forecast run is not successful: {forecast_runs}")
forecast_run_id = forecast_runs[0]["run_id"]
forecast_rows = (
    sb.table("forecast_daily")
    .select(
        "province_id,target_date,forecast_horizon_days,regression_model_name,"
        "regression_run_id,classifier_model_name,classifier_run_id,"
        "classification_source,class_probabilities,fallback_used,forecast_run_id"
    )
    .eq("forecast_run_id", forecast_run_id)
    .order("province_id")
    .order("forecast_horizon_days")
    .execute()
    .data
    or []
)
expected_forecast_rows = PRODUCTION_REQUIRED_PROVINCES * FORECAST_HORIZON_DAYS
if len(forecast_rows) != expected_forecast_rows:
    raise RuntimeError(f"Expected {expected_forecast_rows} forecasts, found {len(forecast_rows)}")
if any(
    row["regression_model_name"] != REGRESSION_MODEL_NAME
    or row["regression_run_id"] != run_id
    or row["fallback_used"]
    for row in forecast_rows
):
    raise RuntimeError("Forecast readback is not using the new residual LightGBM run")
d1_rows = [row for row in forecast_rows if row["forecast_horizon_days"] == 1]
if len(d1_rows) != PRODUCTION_REQUIRED_PROVINCES:
    raise RuntimeError(f"Expected 20 D+1 forecasts, found {len(d1_rows)}")
if any(
    row["classifier_model_name"] != CLASSIFICATION_MODEL_NAME
    or row["classifier_run_id"] != run_id
    or row["classification_source"] != "active_classifier"
    or row["class_probabilities"] is None
    or row["fallback_used"]
    for row in d1_rows
):
    raise RuntimeError("D+1 readback is not using the active pooled Random Forest artifact")

run_summary.update({
    "registered": True,
    "activated": True,
    "active_rows": len(active_for_run),
    "forecast": forecast_response_json,
    "forecast_run_id": forecast_run_id,
    "forecast_rows": len(forecast_rows),
    "d1_active_classifier_rows": len(d1_rows),
    "approved_code_sha": APPROVED_CODE_SHA,
})
summary_path.write_text(
    json.dumps(_json_safe(run_summary), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(_json_safe(run_summary), ensure_ascii=False, indent=2))


In [ ]:
# 16. Download the auditable v5.6.2 Production artifact bundle
import shutil
from google.colab import files

archive_base = Path(f"pm25_residual_dual_artifacts_{run_id}")
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=config.artifact_directory / run_id,
)
print("ZIP ready:", archive_path)
files.download(archive_path)
